In [1]:
import py3Dmol # pip3 install py3Dmol
from pao.io import parse_pao_file, PaoFile
from pathlib import Path
import numpy as np

In [2]:
def coords2xyz(f: PaoFile):
    lattice = " ".join(str(x) for x in f.cell.flatten())
    result = [f"{len(f.atom2kind)}", f'Lattice = "{lattice}"']
    for c, k in zip(f.coords, f.atom2kind):
        result.append(f"{k} {c[0]} {c[1]} {c[2]}")
    return "\n".join(result)

In [3]:
def yzx2xyz(yzx):
    assert yzx.size == 3
    return np.array([yzx[2], yzx[0], yzx[1]])

In [4]:
def make_arrow(view, pos, vec, color):
    a = pos
    b = pos + vec
    view.addArrow({
        "start": {"x":a[0], "y":a[1], "z":a[2]},
        "end": {"x":b[0], "y":b[1], "z":b[2]},
        "radius": 0.1,
        "color": color,
    });

In [5]:
filename = "../../../pao_ml_equivar/2H2O_rotations/phi_00/2H2O_pao42-1_0.pao"

f = parse_pao_file(Path(filename))
assert f.kinds["H"].prim_basis_name == "DZVP-MOLOPT-GTH"

view = py3Dmol.view(data=coords2xyz(f), format=".xyz")
view.setStyle({'stick':{'radius':0.1}, 'sphere':{'radius':0.3}})
view.addUnitCell()

for iatom in range(len(f.atom2kind)):
    pos = f.coords[iatom]
    xblock = f.xblocks[iatom]
    if f.atom2kind[iatom] == "H":
        for j in range(xblock.shape[0]):
            vec = yzx2xyz(xblock[j][2:5]) # p-shell of DZVP-MOLOPT-GTH for Hydrogen
            make_arrow(view, pos=pos, vec=+vec, color="green")
            make_arrow(view, pos=pos, vec=-vec, color="green")
    # elif f.atom2kind[iatom] == "O":
    #     for j in range(xblock.shape[0]):
    #         vec1 = yzx2xyz(xblock[j][2:5]) # 1st p-shell of DZVP-MOLOPT-GTH for Oxygen
    #         make_arrow(view, pos=pos, vec=+vec1, color="yellow")
    #         make_arrow(view, pos=pos, vec=-vec1, color="yellow")
    #         vec2 = yzx2xyz(xblock[j][5:8]) # 2nd p-shell of DZVP-MOLOPT-GTH for Oxygen
    #         make_arrow(view, pos=pos, vec=+vec2, color="blue")
    #         make_arrow(view, pos=pos, vec=-vec2, color="blue")
    
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.